# Data Mining Project - Phase 1
**Group #:** 5

**Members:** Adam Wajda, Bradley Krawchyk, Ekam Lally, Tanishk Sharma

## 1. Problem Definition

Grocery retailers process large numbers of transactions involving thousands of products. This makes it difficult to determine which items should be placed near one another, bundled, recommended, or included in discount planning. Decisions based only on individual product popularity can overlook complementary purchasing relationships and lead to missed cross-selling opportunities.

This project addresses a retail customer market-analysis problem by identifying products and product categories that are purchased together more frequently than expected and comparing these patterns between discounted and non-discounted baskets. The findings can support product placement, bundles, cross-selling, and discount planning. The project will identify associations in the historical data rather than claim that discounts cause purchases.

## 2. Data Collection

The data comes from the public **dunnhumby - The Complete Journey** grocery dataset, which contains anonymized household purchases from one retailer.

- Source description: https://www.dunnhumby.com/source-files/
- CSV distribution used: https://www.kaggle.com/datasets/frtgnn/dunnhumby-the-complete-journey

Three logical datasets are used. `product.csv` was retained from the public source. The transaction and date datasets were generated from the source CSVs. Because the complete transaction file was too large for this course project, a fixed-seed random sample of **20,000 distinct baskets** was selected using seed **74010**. Every receipt line belonging to a selected basket was retained so that basket contents were not distorted. The transaction dataset is stored in three smaller CSV files only for reliable Drive storage; the notebook combines them into one dataset immediately after loading.

### Dataset summaries

| Logical dataset | Collection or generation method | Records | Attributes | Purpose |
|---|---|---:|---:|---|
| `transactions_sample` (three CSV storage parts) | Generated from `transaction_data.csv` by randomly sampling 20,000 complete baskets | 190,161 | 12 | Supplies households, baskets, products, quantities, sales, stores, time, and recorded discounts |
| `product.csv` | Downloaded unchanged from the public source | 92,353 | 7 | Supplies product department, brand, commodity, sub-commodity, manufacturer, and size information |
| `date_dimension.csv` | Generated from the distinct `DAY` and `WEEK_NO` pairs in the sampled transactions | 708 | 2 | Connects each relative transaction day to its study week |

### Biases and limitations

- **Frequent-shopper and retailer bias:** The source represents frequent shoppers from one retailer, so results should not be generalized to all grocery customers or retailers.
- **Time-period bias:** The transactions cover a particular 102-week period, and product, pricing, or discount patterns may differ today.
- **Sampling bias:** Sampling complete baskets preserves the items within each basket, but it does not preserve every shopping trip made by each household. Households that shop more often also contribute more baskets and have more influence on the results.
- **Discount-measurement limitation:** A basket is classified as discounted when a retail, coupon, or coupon-match discount is recorded. This does not prove that the discount caused the purchase.
- **Anonymization limitation:** Household, basket, store, and product identifiers are anonymized, which protects privacy but limits interpretation of individual products and customers.

No surveys, interviews, or personally identifiable information were collected, so the project does not involve purpose-built human-participant data collection.

### Load the data

Place the required local CSV files in `data/` at the repository root. The portfolio bundle does not redistribute the source data. See [DATA_SETUP.md](../DATA_SETUP.md).


In [ ]:
# Configure local paths for the dataset and generated SQLite warehouse.
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import Image, display

root = Path.cwd()
if not (root / "data").is_dir():
    root = root.parent

PROJECT_DIR = root / "data"
TRANSACTIONS_PART1A_PATH = PROJECT_DIR / "transactions_sample_part1a.csv"
TRANSACTIONS_PART1B_PATH = PROJECT_DIR / "transactions_sample_part1b.csv"
TRANSACTIONS_PART2_PATH = PROJECT_DIR / "transactions_sample_part2.csv"
PRODUCT_PATH = PROJECT_DIR / "product.csv"
DATE_PATH = PROJECT_DIR / "date_dimension.csv"
DATABASE_PATH = PROJECT_DIR / "complete_journey.sqlite"
ERD_PATH = PROJECT_DIR / "warehouse_erd.png"

# Stop with a clear message if a required source file is missing.
required_files = [
    TRANSACTIONS_PART1A_PATH,
    TRANSACTIONS_PART1B_PATH,
    TRANSACTIONS_PART2_PATH,
    PRODUCT_PATH,
    DATE_PATH,
]
missing_files = [path.name for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f'Missing project files: {missing_files}')

print(f'Project folder: {PROJECT_DIR}')

In [ ]:
# Load the three storage parts of the single transaction dataset.
transaction_parts = [
    pd.read_csv(TRANSACTIONS_PART1A_PATH),
    pd.read_csv(TRANSACTIONS_PART1B_PATH),
    pd.read_csv(TRANSACTIONS_PART2_PATH),
]

# Combine the parts and load the other two logical datasets.
transactions = pd.concat(transaction_parts, ignore_index=True)
products = pd.read_csv(PRODUCT_PATH, keep_default_na=False)
dates = pd.read_csv(DATE_PATH)

# Confirm the record and attribute counts directly from the loaded files.
dataset_summary = pd.DataFrame([
    {
        'Logical dataset': 'transactions_sample',
        'Records': len(transactions),
        'Attributes': len(transactions.columns),
    },
    {
        'Logical dataset': 'product.csv',
        'Records': len(products),
        'Attributes': len(products.columns),
    },
    {
        'Logical dataset': 'date_dimension.csv',
        'Records': len(dates),
        'Attributes': len(dates.columns),
    },
])

display(dataset_summary)

### Import the datasets into an SQLite warehouse

The relational warehouse contains one transaction fact table and two dimension tables. `product_id` and `day` are foreign keys in `fact_transactions`. The `analysis_transactions` view joins the three tables into **15 attributes** and **190,161 records**, satisfying the required 10–15 attributes and minimum 10,000 records.

Indexes are created for basket, product, household, day, basket-product, and week queries.

In [ ]:
# Normalize column names and remove WEEK_NO from the fact table because it belongs in dim_date.
fact_transactions = transactions.rename(columns=str.lower).drop(columns=['week_no'])
dim_product = products.rename(columns=str.lower)
dim_date = dates.rename(columns=str.lower)

# Create the relational SQLite warehouse in Google Drive.
with sqlite3.connect(DATABASE_PATH) as connection:
    connection.execute('PRAGMA foreign_keys = ON;')

    # Recreate the warehouse so rerunning this cell produces the same structure.
    connection.executescript('''
        DROP VIEW IF EXISTS analysis_transactions;
        DROP TABLE IF EXISTS fact_transactions;
        DROP TABLE IF EXISTS dim_product;
        DROP TABLE IF EXISTS dim_date;

        CREATE TABLE dim_product (
            product_id INTEGER PRIMARY KEY,
            manufacturer INTEGER NOT NULL,
            department TEXT NOT NULL,
            brand TEXT NOT NULL,
            commodity_desc TEXT NOT NULL,
            sub_commodity_desc TEXT NOT NULL,
            curr_size_of_product TEXT NOT NULL
        );

        CREATE TABLE dim_date (
            day INTEGER PRIMARY KEY,
            week_no INTEGER NOT NULL
        );

        CREATE TABLE fact_transactions (
            transaction_line_id INTEGER PRIMARY KEY AUTOINCREMENT,
            household_key INTEGER NOT NULL,
            basket_id INTEGER NOT NULL,
            day INTEGER NOT NULL,
            product_id INTEGER NOT NULL,
            quantity INTEGER NOT NULL,
            sales_value REAL NOT NULL,
            store_id INTEGER NOT NULL,
            retail_disc REAL NOT NULL,
            trans_time INTEGER NOT NULL,
            coupon_disc REAL NOT NULL,
            coupon_match_disc REAL NOT NULL,
            FOREIGN KEY (product_id) REFERENCES dim_product(product_id),
            FOREIGN KEY (day) REFERENCES dim_date(day)
        );
    ''')

    # Insert the three DataFrames while preserving the declared keys and data types.
    dim_product.to_sql('dim_product', connection, if_exists='append', index=False, chunksize=1000, method='multi')
    dim_date.to_sql('dim_date', connection, if_exists='append', index=False, chunksize=1000, method='multi')
    fact_transactions.to_sql('fact_transactions', connection, if_exists='append', index=False, chunksize=1000, method='multi')

    # Add indexes appropriate for basket assembly, joins, household summaries, and time filtering.
    connection.executescript('''
        CREATE INDEX idx_fact_basket ON fact_transactions(basket_id);
        CREATE INDEX idx_fact_product ON fact_transactions(product_id);
        CREATE INDEX idx_fact_household ON fact_transactions(household_key);
        CREATE INDEX idx_fact_day ON fact_transactions(day);
        CREATE INDEX idx_fact_basket_product ON fact_transactions(basket_id, product_id);
        CREATE INDEX idx_date_week ON dim_date(week_no);

        CREATE VIEW analysis_transactions AS
        SELECT
            f.household_key,
            f.basket_id,
            f.day,
            d.week_no,
            f.store_id,
            f.product_id,
            f.quantity,
            f.sales_value,
            f.retail_disc,
            f.coupon_disc,
            f.coupon_match_disc,
            p.department,
            p.brand,
            p.commodity_desc,
            CASE
                WHEN f.retail_disc <> 0
                  OR f.coupon_disc <> 0
                  OR f.coupon_match_disc <> 0
                THEN 'Discounted'
                ELSE 'Not discounted'
            END AS discount_status
        FROM fact_transactions AS f
        JOIN dim_product AS p ON f.product_id = p.product_id
        JOIN dim_date AS d ON f.day = d.day;
    ''')

    connection.commit()

print(f'SQLite warehouse created: {DATABASE_PATH}')

In [ ]:
# Verify the required record count, attribute count, relationships, and indexes.
with sqlite3.connect(DATABASE_PATH) as connection:
    table_counts = pd.read_sql_query('''
        SELECT 'fact_transactions' AS object_name, COUNT(*) AS records FROM fact_transactions
        UNION ALL
        SELECT 'dim_product', COUNT(*) FROM dim_product
        UNION ALL
        SELECT 'dim_date', COUNT(*) FROM dim_date
        UNION ALL
        SELECT 'analysis_transactions', COUNT(*) FROM analysis_transactions
    ''', connection)

    analysis_columns = connection.execute('PRAGMA table_info(analysis_transactions);').fetchall()
    foreign_key_issues = connection.execute('PRAGMA foreign_key_check;').fetchall()
    indexes = pd.read_sql_query('''
        SELECT name AS index_name, tbl_name AS table_name
        FROM sqlite_master
        WHERE type = 'index' AND name LIKE 'idx_%'
        ORDER BY table_name, index_name
    ''', connection)

display(table_counts)
print(f'Analysis attributes: {len(analysis_columns)}')
print(f'Foreign-key issues: {len(foreign_key_issues)}')
display(indexes)

### Entity-relationship diagram

The ERD shows the primary and foreign keys, the one-to-many relationships from `dim_product` and `dim_date` to `fact_transactions`, and the three warehouse tables used by the 15-attribute analytical view.

In [ ]:
# Display the warehouse ERD when the optional source image is available.
if ERD_PATH.exists():
    display(Image(filename=str(ERD_PATH), width=1100))
else:
    print("ERD diagram not supplied; the warehouse schema is defined above.")


## 3. Data Preparation (Missing and Duplicate Data)

Missingness was assessed in all three warehouse tables, not only with `isna()`. The audit also treats empty or whitespace-only text as missing because the product file was intentionally loaded with `keep_default_na=False`. The source label `UNKNOWN` is retained as an observed category rather than silently recoded: it may describe an unknown commodity, but it is not an empty field. Primary-key uniqueness, exact row repetition, transaction-signature repetition, and foreign-key integrity are checked separately.

For categorical product descriptions, three imputation options were considered:

1. **Overall mode** is reproducible but would assign the most common category or package size to unrelated products.
2. **Group-wise mode** (for example, within commodity) uses local context but still invents a package size and cannot handle products whose hierarchy is also absent.
3. **Explicit `Not specified` category** preserves the fact that the source did not supply a value, avoids fabricated product properties, and remains usable in later grouping and association-rule analysis.

The explicit-category method is selected. The missing fields are nominal descriptors, not continuous measurements; mean/median interpolation is therefore invalid. Missingness is substantial for package size and is plausibly structural (some products are sold individually or by weight), so filling with a typical size could bias item-group comparisons.

In [ ]:
# Read the physical warehouse tables so data quality can be checked at its source.
with sqlite3.connect(DATABASE_PATH) as connection:
    fact_missing = pd.read_sql_query('SELECT * FROM fact_transactions', connection)
    product_missing = pd.read_sql_query('SELECT * FROM dim_product', connection)
    date_missing = pd.read_sql_query('SELECT * FROM dim_date', connection)

warehouse_tables = {
    'fact_transactions': fact_missing,
    'dim_product': product_missing,
    'dim_date': date_missing,
}

NULL_LIKE_TEXT = {'', 'na', 'n/a', 'null', 'none', '?'}

def missing_profile(table_name, frame):
    rows = []
    for column in frame.columns:
        true_nulls = int(frame[column].isna().sum())
        blank_or_sentinel = 0
        if pd.api.types.is_object_dtype(frame[column]) or pd.api.types.is_string_dtype(frame[column]):
            normalized = frame[column].fillna('').astype(str).str.strip().str.lower()
            # Empty strings are counted here only when the original value was non-null.
            blank_or_sentinel = int((normalized.isin(NULL_LIKE_TEXT) & frame[column].notna()).sum())
        total = true_nulls + blank_or_sentinel
        if total:
            rows.append({
                'table': table_name, 'column': column,
                'SQL/Pandas nulls': true_nulls,
                'blank/sentinel text': blank_or_sentinel,
                'total missing': total,
                'missing percent': round(100 * total / len(frame), 2),
            })
    return rows

missing_before = pd.DataFrame([
    row for name, frame in warehouse_tables.items()
    for row in missing_profile(name, frame)
])

if missing_before.empty:
    print('No missing values or blank/sentinel strings were found.')
else:
    display(missing_before)
print(f'Total warehouse records assessed: {sum(map(len, warehouse_tables.values())):,}')

In [ ]:
# Compare defensible methods for the nominal fields before selecting one.
size_nonmissing = product_missing.loc[
    product_missing['curr_size_of_product'].astype(str).str.strip().ne(''),
    'curr_size_of_product'
]

imputation_comparison = pd.DataFrame([
    {
        'Method': 'Overall mode',
        'Example result for missing size': size_nonmissing.mode().iloc[0],
        'Advantage': 'Simple and reproducible',
        'Risk': 'Assigns one popular size to unrelated products',
        'Selected': 'No',
    },
    {
        'Method': 'Commodity-level mode',
        'Example result for missing size': 'Varies by commodity',
        'Advantage': 'Uses product context',
        'Risk': 'Still fabricates a size; unavailable when hierarchy is missing',
        'Selected': 'No',
    },
    {
        'Method': 'Explicit missing category',
        'Example result for missing size': 'Not specified',
        'Advantage': 'Preserves uncertainty and supports categorical analysis',
        'Risk': 'Creates a separate category',
        'Selected': 'Yes',
    },
])
display(imputation_comparison)

# Replace only genuinely empty nominal descriptors; retain source-provided UNKNOWN labels.
product_clean = product_missing.copy()
product_text_columns = ['department', 'commodity_desc', 'sub_commodity_desc', 'curr_size_of_product']
for column in product_text_columns:
    empty = product_clean[column].isna() | product_clean[column].astype(str).str.strip().eq('')
    product_clean.loc[empty, column] = 'Not specified'

missing_after = pd.DataFrame(missing_profile('dim_product_clean', product_clean))
print(f'Values imputed with "Not specified": {(product_clean[product_text_columns] == "Not specified").sum().sum():,}')
print(f'Missing values remaining in cleaned product fields: {0 if missing_after.empty else int(missing_after["total missing"].sum())}')

### Duplicate investigation

An exact duplicate means every source attribute is repeated. For the fact table, the warehouse-generated `transaction_line_id` is excluded from this test because it is unique by construction. A second, stricter business-signature check uses household, basket, day, product, store, time, quantity, sales, and discount values. Repeated products within a basket are not removed merely because they share a basket and product: separate scan lines can be legitimate. Dimension primary keys and warehouse foreign keys are also validated.

In [ ]:
fact_source_columns = [c for c in fact_missing.columns if c != 'transaction_line_id']
transaction_signature = [
    'household_key', 'basket_id', 'day', 'product_id', 'store_id', 'trans_time',
    'quantity', 'sales_value', 'retail_disc', 'coupon_disc', 'coupon_match_disc'
]

duplicate_audit = pd.DataFrame([
    {'Test': 'Fact exact duplicates (surrogate key excluded)',
     'Rows flagged': int(fact_missing.duplicated(subset=fact_source_columns, keep=False).sum())},
    {'Test': 'Fact repeated full transaction signatures',
     'Rows flagged': int(fact_missing.duplicated(subset=transaction_signature, keep=False).sum())},
    {'Test': 'Duplicate product primary keys',
     'Rows flagged': int(product_clean.duplicated(subset=['product_id'], keep=False).sum())},
    {'Test': 'Duplicate date primary keys',
     'Rows flagged': int(date_missing.duplicated(subset=['day'], keep=False).sum())},
])

with sqlite3.connect(DATABASE_PATH) as connection:
    foreign_key_issues = connection.execute('PRAGMA foreign_key_check').fetchall()

display(duplicate_audit)
print(f'Foreign-key violations: {len(foreign_key_issues)}')

# Remove only proven duplicates. None are expected, but this makes the operation reproducible.
fact_clean = fact_missing.drop_duplicates(subset=fact_source_columns, keep='first').copy()
product_clean = product_clean.drop_duplicates(subset=['product_id'], keep='first').copy()
date_clean = date_missing.drop_duplicates(subset=['day'], keep='first').copy()

duplicate_rows_removed = (
    len(fact_missing) - len(fact_clean)
    + len(product_missing) - len(product_clean)
    + len(date_missing) - len(date_clean)
)
print(f'Duplicate rows removed: {duplicate_rows_removed:,}')

In [ ]:
# Persist the cleaned dimension and verify that row counts and relationships are preserved.
with sqlite3.connect(DATABASE_PATH) as connection:
    connection.execute('PRAGMA foreign_keys = ON')
    connection.executemany(
        '''UPDATE dim_product
           SET department = ?, commodity_desc = ?, sub_commodity_desc = ?, curr_size_of_product = ?
           WHERE product_id = ?''',
        product_clean[['department', 'commodity_desc', 'sub_commodity_desc',
                       'curr_size_of_product', 'product_id']].itertuples(index=False, name=None)
    )
    connection.commit()
    final_product = pd.read_sql_query('SELECT * FROM dim_product', connection)
    final_fact_count = connection.execute('SELECT COUNT(*) FROM fact_transactions').fetchone()[0]
    final_view_count = connection.execute('SELECT COUNT(*) FROM analysis_transactions').fetchone()[0]
    final_fk_issues = connection.execute('PRAGMA foreign_key_check').fetchall()

validation = pd.DataFrame([
    {'Check': 'Fact row count preserved', 'Result': final_fact_count, 'Expected': len(fact_missing), 'Pass': final_fact_count == len(fact_missing)},
    {'Check': 'Analysis-view row count preserved', 'Result': final_view_count, 'Expected': len(fact_missing), 'Pass': final_view_count == len(fact_missing)},
    {'Check': 'Product row count preserved', 'Result': len(final_product), 'Expected': len(product_missing), 'Pass': len(final_product) == len(product_missing)},
    {'Check': 'Missing product descriptors remaining', 'Result': sum(r['total missing'] for r in missing_profile('final', final_product)), 'Expected': 0, 'Pass': not missing_profile('final', final_product)},
    {'Check': 'Foreign-key violations', 'Result': len(final_fk_issues), 'Expected': 0, 'Pass': len(final_fk_issues) == 0},
])
display(validation)
assert validation['Pass'].all(), 'A post-cleaning validation check failed.'
print('Missing-value and duplicate-data preparation completed successfully.')

### Decision and result

The audit found no numeric nulls, no duplicate transaction records, no duplicate dimension keys, and no orphaned foreign keys. It did identify blank nominal product metadata: 15 department values, 15 commodity descriptions, 15 sub-commodity descriptions, and 30,607 current-size descriptions. These were imputed as `Not specified` rather than with a mode, because a modal category or package size would create false product information. The 52 literal `UNKNOWN` commodity labels were retained because they are explicit source values rather than empty data.

No rows were deleted: both exact and full transactional-signature duplicate tests returned zero. The cleaned dimension was written back without changing transaction, product, or analysis-view record counts, and the final assertions confirm that the targeted missing fields are resolved and all relationships remain valid.

## 4. Data Preparation (Outliers)

Not every extreme value is an error. Large purchases, expensive items, and big coupons are all real, and discounted lines are unusual by design, so deleting them would remove what the project is studying.

Plan: run five detection methods, compare them, inspect what they flag, pick a final set, and fix only the noise.

### Scope

Only the five numeric measures are tested. Identifiers are meaningless as outliers, `day` and `week_no` are fixed by the study period, and rare text categories are a frequency issue.

In [ ]:
# Currently reads the warehouse directly would be using Tanishk's part.
# As: outlier_input = whatever_the_name_is() instead of with sqlite3 line and the line below it
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

RANDOM_SEED = 74010 # for forest

with sqlite3.connect(DATABASE_PATH) as connection:
    outlier_input = pd.read_sql_query('SELECT * FROM analysis_transactions', connection)

MEASURE_COLUMNS = ['quantity', 'sales_value', 'retail_disc', 'coupon_disc', 'coupon_match_disc']
CURRENCY_COLUMNS = ['sales_value', 'retail_disc', 'coupon_disc', 'coupon_match_disc']
EXCLUDED_COLUMNS = ['household_key', 'basket_id', 'day', 'week_no', 'store_id', 'product_id']

print(f'Records: {len(outlier_input):,}    Attributes: {outlier_input.shape[1]}')
print(f'Measures tested for outliers: {MEASURE_COLUMNS}')
print(f'Identifier and time attributes excluded: {EXCLUDED_COLUMNS}')

In [ ]:
# Profile the five measures before applying any detection method.
outlier_profile = outlier_input[MEASURE_COLUMNS].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999]).T
outlier_profile['skew'] = outlier_input[MEASURE_COLUMNS].skew()
outlier_profile['kurtosis'] = outlier_input[MEASURE_COLUMNS].kurt()
outlier_profile['distinct_values'] = [outlier_input[column].nunique() for column in MEASURE_COLUMNS]
outlier_profile['pct_at_most_common_value'] = [
    round((outlier_input[column] == outlier_input[column].mode()[0]).mean() * 100, 2)
    for column in MEASURE_COLUMNS
]

display(outlier_profile.round(3))

In [ ]:
# Visualise the two measures that drive the analysis: units purchased and money spent.
figure, axes = plt.subplots(2, 2, figsize=(13, 7))

axes[0, 0].hist(outlier_input['quantity'], bins=60)
axes[0, 0].set_yscale('log')
axes[0, 0].set_title('quantity - distribution (log frequency)')
axes[0, 0].set_xlabel('quantity')

axes[0, 1].boxplot(outlier_input['quantity'], vert=False)
axes[0, 1].set_title('quantity - boxplot')

axes[1, 0].hist(outlier_input['sales_value'], bins=60)
axes[1, 0].set_yscale('log')
axes[1, 0].set_title('sales_value - distribution (log frequency)')
axes[1, 0].set_xlabel('sales_value')

axes[1, 1].boxplot(outlier_input['sales_value'], vert=False)
axes[1, 1].set_title('sales_value - boxplot')

figure.tight_layout()
plt.show()

The charts: `quantity` has two separate humps with a clear gap between them, so some records use a different scale. `sales_value` has one smooth tail, which is normal for spending.

The problem: most values are low (79% buy one unit, 99% have no coupon), but a few extreme values inflate the averages and leave several columns with identical quartiles.

### Univariate detection methods

* Z-score - flags values >3 standard deviations from the mean. Simple, but distorted by the outliers it looks for.
* Tukey IQR - flags values >1.5x the interquartile range beyond Q1/Q3. Needs the middle 50% to have some spread.
* Modified z-score (MAD)- uses the median and median absolute deviation, threshold 3.5. Most resistant to extremes, but fails when the MAD is zero.

The table also prints each test's calculated values, which explain why the counts differ.

In [ ]:
# Apply the three rules and record the parameters
outlier_flags = pd.DataFrame(index=outlier_input.index)
parameter_rows = []

for column in MEASURE_COLUMNS:
    values = outlier_input[column]

    # Method 1: z-score.
    mean_value, std_value = values.mean(), values.std()
    z_scores = (values - mean_value) / std_value if std_value > 0 else pd.Series(0.0, index=values.index)
    z_flag = z_scores.abs() > 3

    # Method 2: Tukey IQR rule.
    q1, q3 = values.quantile(0.25), values.quantile(0.75)
    iqr = q3 - q1
    lower_fence, upper_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    iqr_flag = (values < lower_fence) | (values > upper_fence)

    # Method 3: modified z-score based on the median absolute deviation.
    median_value = values.median()
    mad_value = (values - median_value).abs().median()
    if mad_value > 0:
        modified_z = 0.6745 * (values - median_value) / mad_value
        mad_flag = modified_z.abs() > 3.5
        mad_status = 'Usable'
    else:
        mad_flag = pd.Series(False, index=values.index)
        mad_status = 'Undefined (MAD = 0)'

    outlier_flags[f'z_{column}'] = z_flag
    outlier_flags[f'iqr_{column}'] = iqr_flag
    outlier_flags[f'mad_{column}'] = mad_flag

    parameter_rows.append({
        'measure': column,
        'mean': round(mean_value, 3),
        'std_dev': round(std_value, 3),
        'Q1': q1,
        'Q3': q3,
        'IQR': round(iqr, 3),
        'IQR_lower_fence': round(lower_fence, 3),
        'IQR_upper_fence': round(upper_fence, 3),
        'median': median_value,
        'MAD': mad_value,
        'MAD_status': mad_status,
        'z_flagged': int(z_flag.sum()),
        'iqr_flagged': int(iqr_flag.sum()),
        'mad_flagged': int(mad_flag.sum()),
    })

univariate_parameters = pd.DataFrame(parameter_rows)
display(univariate_parameters)

# Combine each method across all five measures: a record is flagged if any measure triggers it.
z_score_outliers = outlier_flags[[f'z_{column}' for column in MEASURE_COLUMNS]].any(axis=1)
iqr_outliers = outlier_flags[[f'iqr_{column}' for column in MEASURE_COLUMNS]].any(axis=1)
mad_outliers = outlier_flags[[f'mad_{column}' for column in MEASURE_COLUMNS]].any(axis=1)

for label, flag in [('Z-score', z_score_outliers), ('IQR', iqr_outliers), ('Modified z-score', mad_outliers)]:
    print(f'{label}: {int(flag.sum()):,} records flagged ({flag.mean() * 100:.2f}% of the dataset)')

Why they disagree:

* IQR - Q1 and Q3 are both 1 for `quantity`, so every line that didn't buy exactly one unit is flagged. Same on the coupon columns: every coupon used is called an error.
* Modified z-score - MAD is zero for three of five measures, so it can't be calculated. Where it can, it flags half the dataset.
* Z-score - the standard deviation of `quantity` is over 1,000 because of the extremes it should be catching, so its limit is too wide and some of the largest records slip through. This is called masking.

### Multivariate detection methods

These test one column at a time, so they miss records that are normal per column but odd as a combination. Two methods use all five measures together:

* Isolation Forest - scores how easily random splits separate a record. Finds globally rare records.
* Local Outlier Factor (LOF) - compares how crowded a record's area is with its neighbours'.

Both use 1% contamination so the lists are the same size. LOF can take a minute.

In [ ]:
# Standardise the measures so that both detectors treat the five columns comparably.
scaled_measures = StandardScaler().fit_transform(outlier_input[MEASURE_COLUMNS])

# Method 4: Isolation Forest.
isolation_forest = IsolationForest(
    n_estimators=200,
    contamination=0.01,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
isolation_outliers = pd.Series(isolation_forest.fit_predict(scaled_measures) == -1, index=outlier_input.index)

# Method 5: Local Outlier Factor.
local_outlier_factor = LocalOutlierFactor(n_neighbors=25, contamination=0.01, n_jobs=-1)
lof_outliers = pd.Series(local_outlier_factor.fit_predict(scaled_measures) == -1, index=outlier_input.index)

print(f'Isolation Forest: {int(isolation_outliers.sum()):,} records flagged')
print(f'Local Outlier Factor: {int(lof_outliers.sum()):,} records flagged')

### Comparing the methods

The methods are compared by how many records each one flagged and by how much they agree. Agreement is the share of records flagged by both methods out of the records flagged by either one.

In [ ]:
# Compare the five methods by volume and by pairwise agreement.
detection_methods = {
    'Z-score (|z| > 3)': z_score_outliers,
    'Tukey IQR (1.5 x IQR)': iqr_outliers,
    'Modified z-score (MAD)': mad_outliers,
    'Isolation Forest (1%)': isolation_outliers,
    'Local Outlier Factor (1%)': lof_outliers,
}

method_summary = pd.DataFrame([
    {
        'Method': name,
        'Type': 'Univariate' if index < 3 else 'Multivariate',
        'Records flagged': int(flag.sum()),
        'Percent of dataset': round(flag.mean() * 100, 2),
    }
    for index, (name, flag) in enumerate(detection_methods.items())
])
display(method_summary)

# Pairwise agreement: shared flags as a percentage of the combined flags of both methods.
method_names = list(detection_methods)
agreement_matrix = pd.DataFrame(index=method_names, columns=method_names, dtype=float)
for first in method_names:
    for second in method_names:
        first_flag, second_flag = detection_methods[first], detection_methods[second]
        union_size = (first_flag | second_flag).sum()
        agreement_matrix.loc[first, second] = round((first_flag & second_flag).sum() / union_size * 100, 1) if union_size else np.nan

print('Pairwise agreement between methods (% of combined flags shared):')
display(agreement_matrix)

Counts range from 1% to over half the dataset, and overlap is low for every pair. The two multivariate methods used identical data and budgets but share almost nothing.

Isolation Forest catches the huge `quantity` records. LOF catches none of them, because over a thousand similar points look like a normal crowded area to it. Its duplicate-value warning means the same thing.

So the methods can say what is statistically unusual, but not what is wrong. Next step is to look at the records.

### Diagnosing the flagged records

Flagged records are profiled by department and measure. A real data problem should show something in common.

In [ ]:
# Profile the records flagged by each method to see what they have in common.
diagnosis_rows = []
for name, flag in detection_methods.items():
    flagged = outlier_input[flag]
    if flagged.empty:
        continue
    department_counts = flagged['department'].value_counts()
    top_department = department_counts.index[0].strip() or '(blank)'
    diagnosis_rows.append({
        'Method': name,
        'Records flagged': len(flagged),
        'Median quantity': flagged['quantity'].median(),
        'Max quantity': flagged['quantity'].max(),
        'Median sales_value': round(flagged['sales_value'].median(), 2),
        'Most common department': top_department,
        'Share in that department': f"{department_counts.iloc[0] / len(flagged) * 100:.1f}%",
    })

display(pd.DataFrame(diagnosis_rows))

In [ ]:
# Scan the quantity scale for a natural break between ordinary and extreme values.
quantity_bands = pd.cut(
    outlier_input['quantity'],
    bins=[-0.001, 0, 1, 5, 10, 25, 50, 100, 500, 1000, 5000, 10000, 20000, 40000],
)
quantity_band_counts = quantity_bands.value_counts().sort_index().rename('records').to_frame()
quantity_band_counts['percent'] = (quantity_band_counts['records'] / len(outlier_input) * 100).round(3)
display(quantity_band_counts)

In [ ]:
# Examine the records above the empty band, and check where the masked z-score records sit.
extreme_quantity = outlier_input[outlier_input['quantity'] > 100]

print(f'Records with quantity above 100: {len(extreme_quantity):,}')
print(f'Highest quantity outside those records: {outlier_input.loc[outlier_input["quantity"] <= 100, "quantity"].max()}')
print()
print('Departments containing these records:')
display(extreme_quantity['department'].str.strip().value_counts().to_frame('records'))
print('Commodities containing these records:')
display(extreme_quantity['commodity_desc'].str.strip().value_counts().head(3).to_frame('records'))

implied_unit_price = extreme_quantity['sales_value'] / (extreme_quantity['quantity'] / 1000)
print('Implied price per 1,000 recorded units:')
display(implied_unit_price.describe().round(2).to_frame('implied_price'))

quantity_z_flag = outlier_flags['z_quantity']
print(f'Extreme-quantity records missed by the z-score rule: {int((~quantity_z_flag & (outlier_input["quantity"] > 100)).sum())}')

In [ ]:
zero_quantity_count = int((outlier_input['quantity'] == 0).sum())
zero_quantity_and_zero_sales = int(((outlier_input['quantity'] == 0) & (outlier_input['sales_value'].abs() < 0.005)).sum())
zero_quantity_with_discount = int((
    (outlier_input['quantity'] == 0)
    & (outlier_input[['retail_disc', 'coupon_disc', 'coupon_match_disc']].abs().sum(axis=1) >= 0.005)
).sum())

print(f'Lines recording zero units purchased: {zero_quantity_count:,}')
print(f'  of which also record zero sales value: {zero_quantity_and_zero_sales:,}')
print(f'  of which carry a discount adjustment:  {zero_quantity_with_discount:,}')

residue_mask = pd.Series(False, index=outlier_input.index)
for column in CURRENCY_COLUMNS:
    residue_mask |= (outlier_input[column] != 0) & (outlier_input[column].abs() < 0.005)

print(f'\nRecords holding floating-point residue in a currency column: {int(residue_mask.sum())}')
display(outlier_input.loc[residue_mask, ['quantity'] + CURRENCY_COLUMNS])

print(f'Discount larger in absolute value than sales_value: {int((outlier_input["retail_disc"].abs() > outlier_input["sales_value"]).sum()):,}')
print(f'Zero sales_value with a positive quantity:          {int(((outlier_input["sales_value"] < 0.005) & (outlier_input["quantity"] > 0)).sum()):,}')
print(f'Discounts recorded as positive values:              {int((outlier_input[["retail_disc", "coupon_disc", "coupon_match_disc"]] > 0).to_numpy().sum())}')

Findings:

* Huge quantities are a unit problem, not an error. Nothing falls between 100 and 1,000, about 1,600 records sit above 1,000, and the highest below the gap is 74. All are `KIOSK-GAS` or `MISC SALES TRAN`. Read as thousandths of a unit, the price works out to about $2.35, which matches fuel sold by the gallon.
* Masking confirmed: 77 of these are missed by the z-score.
* Everything else flagged is normal shopping – big but believable purchases, expensive items, large coupons.
* Zero-quantity lines: about 1,000 records with no units purchased, so no item in the basket.
* Floating-point noise: 6 records store a currency value as a tiny fraction instead of zero, including 4 positive discounts.

Checked and kept: discounts larger than sales value, and zero sales value with positive quantity. Both are normal, since `sales_value` is what was paid and `retail_disc` is the cut from the shelf price.

### Final outlier set

Based on these findings, not on one method. The methods disagree too much to pick one, and the records that matter here are extreme by nature. They still showed where to look.

| Category | Rule | What it is | Treatment |
|---|---|---|---|
| A. Unit-of-measure lines | `quantity` above 100 | Real fuel and kiosk purchases on a different scale | Kept and labelled, left out of the basket subset |
| B. Zero-quantity lines | `quantity` equal to 0 | Records with no purchased item | Deleted as noise |
| C. Floating-point noise | Currency values below half a cent but not zero | Arithmetic noise | Rounded to two decimals |
| D. Genuine extremes | Everything else flagged | Bulk purchases, expensive items, large coupons | Kept unchanged |

The 100-unit limit comes from the empty gap. Anything between 75 and 1,000 selects the same records.

### Mitigating the noise

* Category C - rounded to 2 decimals. Money is only meaningful to the cent, so nothing real changes.
* Category B - deleted. Quantity is zero, not missing, so there is nothing to fill in.
* Category A - kept and labelled with `line_type`, but excluded from the basket subset. Deleting it would lose real spending, and rescaling it would assume a conversion the data suggests but does not prove.

In [ ]:
# Category C: correct the floating-point residue in the currency columns.
transactions_outliers_handled = outlier_input.copy()

currency_before = transactions_outliers_handled[CURRENCY_COLUMNS].copy()
transactions_outliers_handled[CURRENCY_COLUMNS] = transactions_outliers_handled[CURRENCY_COLUMNS].round(2) + 0.0
residue_corrected = int(
    (currency_before.to_numpy() != transactions_outliers_handled[CURRENCY_COLUMNS].to_numpy()).any(axis=1).sum()
)
positive_discounts_remaining = int(
    (transactions_outliers_handled[['retail_disc', 'coupon_disc', 'coupon_match_disc']] > 0).to_numpy().sum()
)

print(f'Category C - records corrected by rounding to the cent: {residue_corrected}')
print(f'Category C - impossible positive discounts remaining: {positive_discounts_remaining}')

# Categories A and B: label every line so the treatment is recorded in the data itself.
EXTREME_QUANTITY_CUT = 100
line_type = pd.Series('Standard merchandise line', index=transactions_outliers_handled.index)
line_type[transactions_outliers_handled['quantity'] > EXTREME_QUANTITY_CUT] = 'Non-merchandise / unit-of-measure line'
line_type[transactions_outliers_handled['quantity'] == 0] = 'Zero-quantity (empty) line'
transactions_outliers_handled['line_type'] = line_type

display(transactions_outliers_handled['line_type'].value_counts().to_frame('records'))

In [ ]:
# Category B: remove the zero-quantity lines, which carry no purchased item.
zero_quantity_lines = int((transactions_outliers_handled['line_type'] == 'Zero-quantity (empty) line').sum())
transactions_outliers_handled = (
    transactions_outliers_handled[transactions_outliers_handled['line_type'] != 'Zero-quantity (empty) line']
    .reset_index(drop=True)
)

# Build the subset used for the market-basket analysis
basket_analysis_lines = (
    transactions_outliers_handled[transactions_outliers_handled['line_type'] == 'Standard merchandise line']
    .reset_index(drop=True)
)

print(f'Records before outlier mitigation:            {len(outlier_input):,}')
print(f'Category B records removed as noise:          {zero_quantity_lines:,}')
print(f'Prepared table (labelled, noise removed):     {len(transactions_outliers_handled):,}')
print(f'Basket-analysis subset (standard lines only): {len(basket_analysis_lines):,}')

In [ ]:
# Verify mitgation
before_after = pd.concat([
    outlier_input[MEASURE_COLUMNS].describe().T[['mean', 'std', 'max']].add_prefix('before_'),
    basket_analysis_lines[MEASURE_COLUMNS].describe().T[['mean', 'std', 'max']].add_prefix('after_'),
], axis=1)
display(before_after.round(3))

coverage = pd.DataFrame([
    {
        'Measure': 'Distinct baskets',
        'Before': outlier_input['basket_id'].nunique(),
        'After': basket_analysis_lines['basket_id'].nunique(),
    },
    {
        'Measure': 'Distinct households',
        'Before': outlier_input['household_key'].nunique(),
        'After': basket_analysis_lines['household_key'].nunique(),
    },
    {
        'Measure': 'Distinct products',
        'Before': outlier_input['product_id'].nunique(),
        'After': basket_analysis_lines['product_id'].nunique(),
    },
    {
        'Measure': 'Total sales value',
        'Before': round(outlier_input['sales_value'].sum(), 2),
        'After': round(basket_analysis_lines['sales_value'].sum(), 2),
    },
])
coverage['Retained %'] = (coverage['After'] / coverage['Before'] * 100).round(2)
display(coverage)

discount_balance = pd.concat([
    outlier_input['discount_status'].value_counts(normalize=True).mul(100).round(2).rename('Before %'),
    basket_analysis_lines['discount_status'].value_counts(normalize=True).mul(100).round(2).rename('After %'),
], axis=1)
display(discount_balance)

`quantity` mean drops 95 to 1.3 and standard deviation 1,113 to under 1, purely from separating the fuel lines. `sales_value` max stays at 200, since the expensive purchases were kept.

Baskets drop 20,000 to 18,389. Those were single-line fuel or empty trips, and a one-item basket makes no product pairs. The discounted/non-discounted split is unchanged, which is the key check.

### Saving the prepared data

Written back to the warehouse with indexes. `fact_transactions` is untouched, so the original data is still there.

In [ ]:
# Write the prepared table back to the warehouse and index it for the analysis queries.
with sqlite3.connect(DATABASE_PATH) as connection:
    transactions_outliers_handled.to_sql(
        'analysis_transactions_prepared',
        connection,
        if_exists='replace',
        index=False,
        chunksize=1000,
        method='multi',
    )

    connection.executescript('''
        DROP INDEX IF EXISTS idx_prepared_basket;
        DROP INDEX IF EXISTS idx_prepared_product;
        DROP INDEX IF EXISTS idx_prepared_basket_product;
        DROP INDEX IF EXISTS idx_prepared_line_type;

        CREATE INDEX idx_prepared_basket ON analysis_transactions_prepared(basket_id);
        CREATE INDEX idx_prepared_product ON analysis_transactions_prepared(product_id);
        CREATE INDEX idx_prepared_basket_product ON analysis_transactions_prepared(basket_id, product_id);
        CREATE INDEX idx_prepared_line_type ON analysis_transactions_prepared(line_type);
    ''')
    connection.commit()

    prepared_check = pd.read_sql_query('''
        SELECT line_type, COUNT(*) AS records, ROUND(SUM(sales_value), 2) AS total_sales
        FROM analysis_transactions_prepared
        GROUP BY line_type
        ORDER BY records DESC
    ''', connection)

print('Prepared table written to the warehouse: analysis_transactions_prepared')
display(prepared_check)

## 5. Data Preparation (Formatting)

Formatting is performed after missing-value, duplicate, and outlier treatment so that the final categories describe the analysis-ready records. Numerical measures are discretized with methods suited to their distributions rather than one rule for every field:

- **Quantity uses business-defined intervals** (`1`, `2-3`, `4-5`, `6-10`, `11-100`, and `over 100`). These boundaries preserve common grocery purchase sizes and isolate the unit-of-measure/non-merchandise records identified during outlier analysis.
- **Sales value uses quartiles** because spending is strongly right-skewed. Equal-frequency bins avoid placing nearly every purchase in the lowest interval and produce useful groups for association analysis.
- **Total discount uses fixed dollar bands** (`none`, `up to $1`, `$1-$5`, and `over $5`). Zero is separated because the discount variables are zero-inflated, while the remaining bands are easy for a retailer to interpret.
- **Week number uses four equal-width study periods**. Week is temporal and ordered, so equal-width intervals preserve progression through the observed study period.

The resulting bins are ordinal and are translated to integers in their natural low-to-high order. Nominal fields have no natural ranking, so each is assigned a reproducible alphabetical lookup code. These integer codes are identifiers for categories, not statements that one category is greater than another. Household, basket, store, and product IDs are retained as keys and are not treated as continuous measurements.

In [ ]:
# Use the outlier-prepared data created immediately above.
# The fallback permits this section to be tested independently after the warehouse exists.
if 'transactions_outliers_handled' in globals():
    formatting_data = transactions_outliers_handled.copy()
else:
    with sqlite3.connect(DATABASE_PATH) as connection:
        formatting_data = pd.read_sql_query('SELECT * FROM analysis_transactions', connection)
    formatting_data[['sales_value', 'retail_disc', 'coupon_disc', 'coupon_match_disc']] = (
        formatting_data[['sales_value', 'retail_disc', 'coupon_disc', 'coupon_match_disc']].round(2) + 0.0
    )
    formatting_data['line_type'] = 'Standard merchandise line'
    formatting_data.loc[formatting_data['quantity'] > 100, 'line_type'] = 'Non-merchandise / unit-of-measure line'
    formatting_data = formatting_data.loc[formatting_data['quantity'] != 0].reset_index(drop=True)

formatting_input_count = len(formatting_data)
print(f'Records formatted: {formatting_input_count:,}')
print(f'Input attributes: {formatting_data.shape[1]}')

In [ ]:
# Discretize numerical columns.
quantity_labels = ['1 item', '2-3 items', '4-5 items', '6-10 items', '11-100 items', 'Over 100 units']
formatting_data['quantity_bin'] = pd.cut(
    formatting_data['quantity'],
    bins=[0, 1, 3, 5, 10, 100, float('inf')],
    labels=quantity_labels,
    include_lowest=True,
    ordered=True,
)

sales_labels = ['Low sales', 'Medium sales', 'High sales', 'Very high sales']
formatting_data['sales_value_bin'], sales_edges = pd.qcut(
    formatting_data['sales_value'], q=4, labels=sales_labels,
    retbins=True, duplicates='drop'
)

formatting_data['total_discount'] = -formatting_data[
    ['retail_disc', 'coupon_disc', 'coupon_match_disc']
].sum(axis=1)
discount_labels = ['No discount', 'Up to $1', '$1-$5', 'Over $5']
formatting_data['discount_amount_bin'] = pd.cut(
    formatting_data['total_discount'],
    bins=[-0.001, 0, 1, 5, float('inf')],
    labels=discount_labels,
    include_lowest=True,
    ordered=True,
)

week_labels = ['Early period', 'Early-middle period', 'Late-middle period', 'Late period']
formatting_data['week_period'] = pd.cut(
    formatting_data['week_no'], bins=4, labels=week_labels,
    include_lowest=True, ordered=True
)

bin_counts = pd.concat({
    'quantity_bin': formatting_data['quantity_bin'].value_counts(sort=False),
    'sales_value_bin': formatting_data['sales_value_bin'].value_counts(sort=False),
    'discount_amount_bin': formatting_data['discount_amount_bin'].value_counts(sort=False),
    'week_period': formatting_data['week_period'].value_counts(sort=False),
}).rename('records').reset_index().rename(columns={'level_0':'variable','level_1':'bin'})

print('Sales-value quartile boundaries:', [round(float(value), 2) for value in sales_edges])
display(bin_counts)

In [ ]:
# Translate ordered categories to explicit low-to-high integer codes.
ordinal_mappings = {
    'quantity_bin': {label: code for code, label in enumerate(quantity_labels)},
    'sales_value_bin': {label: code for code, label in enumerate(sales_labels)},
    'discount_amount_bin': {label: code for code, label in enumerate(discount_labels)},
    'week_period': {label: code for code, label in enumerate(week_labels)},
}

for column, mapping in ordinal_mappings.items():
    formatting_data[f'{column}_code'] = formatting_data[column].astype(str).map(mapping).astype('int16')

# Alphabetical coding is deterministic; mapping tables preserve interpretability.
nominal_columns = ['department', 'brand', 'commodity_desc', 'discount_status', 'line_type']
nominal_mappings = {}
for column in nominal_columns:
    categories = sorted(formatting_data[column].astype(str).unique())
    mapping = {category: code for code, category in enumerate(categories)}
    nominal_mappings[column] = mapping
    formatting_data[f'{column}_code'] = formatting_data[column].astype(str).map(mapping).astype('int32')

mapping_summary = pd.DataFrame([
    {'variable': column, 'type': 'Ordinal', 'categories': len(mapping),
     'code range': f'{min(mapping.values())}-{max(mapping.values())}'}
    for column, mapping in ordinal_mappings.items()
] + [
    {'variable': column, 'type': 'Nominal', 'categories': len(mapping),
     'code range': f'{min(mapping.values())}-{max(mapping.values())}'}
    for column, mapping in nominal_mappings.items()
])
display(mapping_summary)

print('Example nominal lookup — brand:', nominal_mappings['brand'])
print('Example ordinal lookup — sales value:', ordinal_mappings['sales_value_bin'])

In [ ]:
# Validate the encoded fields and save the formatted analysis table.
encoded_columns = [
    f'{column}_code' for column in list(ordinal_mappings) + nominal_columns
]

formatting_validation = pd.DataFrame([
    {'check': 'Record count preserved', 'result': len(formatting_data),
     'expected': formatting_input_count, 'pass': len(formatting_data) == formatting_input_count},
    {'check': 'Missing values in generated bins/codes',
     'result': int(formatting_data[[
         'quantity_bin', 'sales_value_bin', 'discount_amount_bin', 'week_period', *encoded_columns
     ]].isna().sum().sum()), 'expected': 0,
     'pass': not formatting_data[[
         'quantity_bin', 'sales_value_bin', 'discount_amount_bin', 'week_period', *encoded_columns
     ]].isna().any().any()},
    {'check': 'Encoded columns use integer data types',
     'result': sum(pd.api.types.is_integer_dtype(formatting_data[c]) for c in encoded_columns),
     'expected': len(encoded_columns),
     'pass': all(pd.api.types.is_integer_dtype(formatting_data[c]) for c in encoded_columns)},
])
display(formatting_validation)
assert formatting_validation['pass'].all(), 'A formatting validation check failed.'

# SQLite does not support pandas categorical dtype, so save their readable text labels.
formatted_for_sql = formatting_data.copy()
for column in ['quantity_bin', 'sales_value_bin', 'discount_amount_bin', 'week_period']:
    formatted_for_sql[column] = formatted_for_sql[column].astype(str)

with sqlite3.connect(DATABASE_PATH) as connection:
    formatted_for_sql.to_sql(
        'analysis_transactions_formatted', connection,
        if_exists='replace', index=False, chunksize=1000, method='multi'
    )
    saved_rows = connection.execute(
        'SELECT COUNT(*) FROM analysis_transactions_formatted'
    ).fetchone()[0]

print(f'Formatted table saved: analysis_transactions_formatted ({saved_rows:,} records)')
print('Formatting validation completed successfully.')

### Formatting result

All analysis-ready records were retained. Four numerical concepts were converted into interpretable ordered bins, and both the readable labels and their ordered integer codes were preserved. Five nominal fields were converted using deterministic lookup dictionaries, with examples printed above. Identifier columns remain unchanged because their numbers identify entities rather than quantities or ranks. The validation confirms that every generated bin and code is complete, all encoded columns have integer data types, and the formatted table contains the expected number of records.